# 🩺 Medical RAG Assistant (Self-Contained Notebook Pipeline)

This notebook contains the **entire Medical RAG Assistant** in a single, clean, self-contained file. It includes page extraction, recursive text chunking, local vector embeddings, local persistent storage, semantic retrieval, and generation with Gemini 2.5 Flash.

### Step 1: Install & Import Dependencies

In [ ]:
# Install dependencies if not already installed:
# !pip install streamlit langchain langchain-community langchain-google-genai chromadb sentence-transformers pypdf python-dotenv

import os
import time
from typing import List
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# Load environment variables from .env file
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

if api_key:
    print("✅ API key configured successfully.")
else:
    print("⚠️ API Key missing! Set GEMINI_API_KEY or GOOGLE_API_KEY in your .env file.")

### Step 2: Define Core Processing Classes

In [ ]:
class MedicalRAGPipeline:
    def __init__(self, api_key: str, db_dir: str = "db"):
        self.api_key = api_key
        self.db_dir = db_dir
        self.embeddings = None
        self.vector_store = None
        self.llm = None
        
        # Initialize local embeddings model
        print("🔄 Loading local embedding model (sentence-transformers)...\n")
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2",
            encode_kwargs={'normalize_embeddings': True}
        )
        
        # Initialize Gemini 2.5 Flash LLM
        self.llm = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash",
            google_api_key=self.api_key,
            temperature=0.1
        )
        print("✅ Embedding model and Gemini LLM successfully initialized.")
        
    def ingest_document(self, pdf_path: str) -> int:
        """Loads, chunks, and indexes a PDF document in ChromaDB."""
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"PDF path '{pdf_path}' not found.")
            
        # 1. Load PDF pages
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        print(f"📄 Loaded {len(documents)} pages from '{pdf_path}'.")
        
        # 2. Split pages into overlapping chunks
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=700,
            chunk_overlap=100,
            add_start_index=True
        )
        chunks = splitter.split_documents(documents)
        print(f"✂️ Split pages into {len(chunks)} overlapping text chunks.")
        
        # 3. Create persistent vector store
        print("💾 Indexing chunks in ChromaDB...")
        self.vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            persist_directory=self.db_dir
        )
        print("✅ Document indexing completed.")
        return len(chunks)
        
    def query(self, user_question: str, k: int = 3) -> dict:
        """Runs similarity retrieval and invokes Gemini for a grounded answer."""
        if not self.vector_store:
            raise ValueError("No document has been ingested yet! Call ingest_document() first.")
            
        start_time = time.time()
        
        # 1. Retrieve top-k chunks with L2 distance score from Chroma
        retrieved_results = self.vector_store.similarity_search_with_score(user_question, k=k)
        
        # 2. De-duplicate identical text chunks to keep the output clean
        seen_content = set()
        unique_docs_with_scores = []
        for doc, score in retrieved_results:
            cleaned_content = doc.page_content.strip()
            if cleaned_content not in seen_content:
                seen_content.add(cleaned_content)
                unique_docs_with_scores.append((doc, score))
                
        # 3. Format the context text block
        context_text = "\n\n".join([
            f"Source: {d.metadata.get('source')} (Page {d.metadata.get('page') + 1}):\n{d.page_content}"
            for d, _ in unique_docs_with_scores
        ])
        
        # 4. Construct grounded system prompt
        system_instruction = (
            "You are an expert Medical AI Assistant designed to answer questions accurately "
            "using only the provided context. Follow these rules strictly:\n"
            "1. Answer the question using ONLY the provided clinical context. Do not use outside medical knowledge.\n"
            "2. Be concise, professional, and clear.\n"
            "3. If the retrieved context does not contain enough information to answer the question, "
            "clearly state: 'I cannot answer this based on the provided document.'\n"
            "4. Never hallucinate, speculate, or make up facts. Your response must be grounded strictly in the source text.\n\n"
            "--- Context ---\n"
            f"{context_text}\n"
            "----------------"
        )
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_instruction),
            ("human", "{question}")
        ])
        
        # 5. Invoke generative LLM
        prompt_messages = prompt.format_messages(question=user_question)
        response = self.llm.invoke(prompt_messages)
        
        latency = time.time() - start_time
        
        return {
            "answer": response.content,
            "sources": unique_docs_with_scores,
            "latency_seconds": latency
        }

### Step 3: Run Ingestion

In [ ]:
# Initialize and run document ingestion
pdf_path = "data/sample_medical.pdf"
pipeline = MedicalRAGPipeline(api_key=api_key)

try:
    num_chunks = pipeline.ingest_document(pdf_path)
    print(f"📄 Ingested {pdf_path} into database successfully. Total text chunks generated: {num_chunks}.")
except Exception as e:
    print(f"❌ Ingestion failed: {e}")

### Step 4: Interactive Clinical Q&A Loop

Run the cell below to start an interactive clinical Q&A session. Enter your medical questions. Type `exit` to quit.

In [ ]:
if not pipeline.vector_store:
    print("⚠️ Please index a document first in Step 3.")
else:
    print("🩺 Medical Q&A Session Started. Type 'exit' to stop.\n")
    
    while True:
        question = input("🩺 Query: ")
        if question.lower().strip() == "exit":
            print("\n👋 Q&A session finished.")
            break
            
        if not question.strip():
            continue
            
        try:
            res = pipeline.query(question, k=3)
            
            print(f"\n💡 Grounded Answer:\n{res['answer']}")
            print(f"\n⏱️ Response Latency: {res['latency_seconds']:.2f} seconds")
            
            print("\n📚 Retrieved Sources & Distance Metrics:")
            for i, (doc, score) in enumerate(res['sources']):
                cosine_sim = 1.0 - (score ** 2) / 2.0
                print(f"  - Reference {i+1}: Page {doc.metadata.get('page') + 1} | L2 Distance: {score:.4f} | Cosine Similarity: {cosine_sim:.4f}")
                print(f"    Content snippet: '{doc.page_content[:150]}...'")
            print("-" * 80 + "\n")
            
        except Exception as e:
            print(f"❌ Error querying: {e}\n")